In [ ]:
!nvidia-smi

Thu Apr 30 07:52:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Installation


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

# MODEL

In [ ]:
!pip install --upgrade torchao

from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit = False,                    # H100 varsa False yap (16-bit, daha kaliteli)
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # Encoder donduruldu → overfit azalir
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 16,                    # 32 → 16, gereksiz kapasiteyi kes
    lora_alpha   = 16,                    # alpha/r = 2 → daha stabil sinyal
    lora_dropout = 0.00,                  # Regularizasyon icin
    bias         = "none",
    random_state = 3407,
    use_rslora   = True,                  # Daha stabil gradient scaling
    loftq_config = None,
)

/tmp/ipykernel_6420/2511097745.py:3: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# Data Rep

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset

HF_DATASET = "SalihHub/blind-assist-tr-image-to-text-QA-style"
raw = load_dataset(HF_DATASET)

class VisionDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img = sample["image"].convert("RGB")
        return {
            "messages": [
                {"role": "system", "content": sample["system"]},
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text",  "text": sample["question"]},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": sample["answer"]}],
                },
            ]
        }

converted_train = VisionDataset(raw["train"])
converted_test  = VisionDataset(raw["test"])

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

CHAT TEMPLATE

In [ ]:
!pip install --upgrade torchao

from unsloth import get_chat_template
processor = get_chat_template(processor, "gemma-4")


EGITIM ONCESI TEST

In [ ]:
FastVisionModel.for_inference(model)

sample    = raw["test"][0]
pil_image = sample["image"].convert("RGB")

messages = [
    {"role": "system", "content": sample["system"]},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": pil_image},   # IMAGE ONCE
            {"type": "text",  "text": sample["question"]},
        ],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(pil_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(processor, skip_prompt=True)
print(f"=== EGITIM ONCESI | Soru: {sample['question']} ===")
_ = model.generate(**inputs, streamer=streamer, max_new_tokens=256,
                   temperature=0.7, top_p=0.95, top_k=64)


EGITIM

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | Max VRAM: {max_memory} GB")

trainer = SFTTrainer(
    model            = model,
    train_dataset    = converted_train,
    eval_dataset     = converted_test,
    processing_class = processor.tokenizer,
    data_collator    = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size  = 32,
        gradient_accumulation_steps  = 4,
        num_train_epochs             = 1,
        # max_steps = 100,                   # Hizli test icin bu satiri ac, usttekini kapat
        max_grad_norm                = 1.0,
        warmup_steps                 = 39,
        learning_rate                = 5e-5,
        logging_steps                = 10,
        save_strategy                = "steps",  # Her 50 stepte checkpoint
        save_steps                   = 100,
        eval_strategy                = "steps",  # Her 50 stepte eval
        eval_steps                   = 100,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 3407,
        output_dir                   = "blind_assist_outputs",
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        report_to                    = "none",
        # Vision icin ZORUNLU:
        remove_unused_columns        = False,
        dataset_text_field           = "",
        dataset_kwargs               = {"skip_prepare_dataset": True},
        max_length                   = 2048,
    ),
)

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Egitim: {trainer_stats.metrics['train_runtime']/60:.1f} dakika")
print(f"Peak VRAM: {used_memory} / {max_memory} GB")

GPU: NVIDIA A100-SXM4-80GB | Max VRAM: 79.251 GB
Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 87,816 | Num Epochs = 1 | Total steps = 687
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 4 x 1) = 128
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss,Validation Loss
100,1.109699,2.822239
200,0.810706,2.826398
300,0.730333,2.831225
400,0.687070,2.845594
500,0.668845,2.841274
600,0.664278,2.835989
687,0.645472,2.845897


Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in blind_assist_outputs/checkpoint-687/tokenizer_config.json.


Egitim: 228.1 dakika
Peak VRAM: 65.764 / 79.251 GB


EGITIM SONRASI TEST

In [ ]:
from unsloth import FastVisionModel
import torch
from transformers import TextStreamer

# Checkpoint'i yükle
model, processor = FastVisionModel.from_pretrained(
    "/content/blind_assist_outputs/checkpoint-687",
    load_in_4bit = False,
)

FastVisionModel.for_inference(model)

# Test görseli
test_sample = raw["test"][5]
test_pil    = test_sample["image"].convert("RGB")

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": test_sample["system"]}],  # string → liste
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_pil},
            {"type": "text",  "text": test_sample["question"]},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize              = True,
    return_tensors        = "pt",
    return_dict           = True,
).to("cuda")

streamer = TextStreamer(processor.tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer       = streamer,
    max_new_tokens = 512,
    temperature    = 0.7,
    top_p          = 0.95,
    top_k          = 64,
)

print(f"\n--- Soru: {test_sample['question']}")
print(f"--- Beklenen: {test_sample['answer']}")

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Görüntünün merkezinde, açık renkli (muhtemelen beyaz) bir hasta yatağı veya muayene koltuğu bulunuyor. Bu koltuğa doğrudan ilerleyebilirsiniz.<turn|>

--- Soru: Hasta koltuğuna nasıl ulaşabilirim?
--- Beklenen: Önünüzde, odanın merkezine yakın bir yerde, beyaz ve kırmızı detaylı bir hasta koltuğu duruyor. Koltuğa doğru düz ilerleyebilirsiniz.


KAYDET

In [ ]:
!pip install -q ai-edge-torch torch transformers huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.6/570.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.0/116.0 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.5 MB/s eta 0:00:00
